<a href="https://colab.research.google.com/github/ayushpaliwal1920/NLP/blob/main/Word2Vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/


cp: cannot stat 'kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d saurabhbadole/game-of-thrones-book-dataset

Dataset URL: https://www.kaggle.com/datasets/saurabhbadole/game-of-thrones-book-dataset
License(s): other
100% 3.71M/3.71M [00:00<00:00, 5.35MB/s]



In [3]:

import zipfile
zip_ref = zipfile.ZipFile('/content/game-of-thrones-book-dataset.zip')
zip_ref.extractall('/content')
zip_ref.close()

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 23.1 MB/s eta 0:00:00


In [5]:
import gensim
import os
from gensim.models import word2vec
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
import nltk
from nltk.tokenize import sent_tokenize


nltk.download('punkt')
nltk.download('punkt_tab')
from gensim.utils import simple_preprocess

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [15]:
story = []

folder_path = "/content"

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
       with open(os.path.join(folder_path, filename), 'r', encoding='latin-1') as f:
            corpus = f.read()

            sentences = sent_tokenize(corpus)

            for sent in sentences:
                tokens = simple_preprocess(sent)
                story.append(tokens)

In [59]:
print(story[:5])

[['clash', 'of', 'kings', 'book', 'two', 'of', 'song', 'of', 'ice', 'and', 'fire', 'by', 'george', 'martin', 'prologue', 'the', 'comet', 'tail', 'spread', 'across', 'the', 'dawn', 'red', 'slash', 'that', 'bled', 'above', 'the', 'crags', 'of', 'dragonstone', 'like', 'wound', 'in', 'the', 'pink', 'and', 'purple', 'sky'], ['the', 'maester', 'stood', 'on', 'the', 'windswept', 'balcony', 'outside', 'his', 'chambers'], ['it', 'was', 'here', 'the', 'ravens', 'came', 'after', 'long', 'flight'], ['their', 'droppings', 'speckled', 'the', 'gargoyles', 'that', 'rose', 'twelve', 'feet', 'tall', 'on', 'either', 'side', 'of', 'him', 'hellhound', 'and', 'wyvern', 'two', 'of', 'the', 'thousand', 'that', 'brooded', 'over', 'the', 'walls', 'of', 'the', 'ancient', 'fortress'], ['when', 'first', 'he', 'came', 'to', 'dragonstone', 'the', 'army', 'of', 'stone', 'grotesques', 'had', 'made', 'him', 'uneasy', 'but', 'as', 'the', 'years', 'passed', 'he', 'had', 'grown', 'used', 'to', 'them']]


In [52]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [53]:
clean_story = []

for sentence in story:
    filtered = [word for word in sentence if word not in stop_words]
    clean_story.append(filtered)

print(clean_story[:5])

[['clash', 'kings', 'book', 'two', 'song', 'ice', 'fire', 'george', 'martin', 'prologue', 'comet', 'tail', 'spread', 'across', 'dawn', 'red', 'slash', 'bled', 'crags', 'dragonstone', 'like', 'wound', 'pink', 'purple', 'sky'], ['maester', 'stood', 'windswept', 'balcony', 'outside', 'chambers'], ['ravens', 'came', 'long', 'flight'], ['droppings', 'speckled', 'gargoyles', 'rose', 'twelve', 'feet', 'tall', 'either', 'side', 'hellhound', 'wyvern', 'two', 'thousand', 'brooded', 'walls', 'ancient', 'fortress'], ['first', 'came', 'dragonstone', 'army', 'stone', 'grotesques', 'made', 'uneasy', 'years', 'passed', 'grown', 'used']]


In [55]:
len(clean_story)

145020

# Model Training||

In [56]:
model = gensim.models.Word2Vec(
    clean_story,
    window = 10,
    min_count=2,
    workers = 4,
    sg = 1 # skip gram
)

In [57]:
model.build_vocab(clean_story) # building vocab

In [58]:
model.train(clean_story,total_examples = model.corpus_count ,epochs = model.epochs) # training

(4396454, 4579390)

In [60]:
model.wv.most_similar("jaime")

[('cersei', 0.7054210305213928),
 ('lannister', 0.6680952906608582),
 ('kingslayer', 0.6625311374664307),
 ('tyrion', 0.6452714800834656),
 ('kevan', 0.6392558813095093),
 ('brienne', 0.6294592618942261),
 ('pod', 0.6263933777809143),
 ('pressing', 0.6158052086830139),
 ('edmure', 0.6153825521469116),
 ('imp', 0.6132999062538147)]

In [61]:
model.wv.most_similar("tywin")


[('lannister', 0.7453066110610962),
 ('kevan', 0.6750599145889282),
 ('pays', 0.664012610912323),
 ('debts', 0.6310077905654907),
 ('kinslaying', 0.6297603249549866),
 ('kingslayer', 0.6285653710365295),
 ('beric', 0.6263079047203064),
 ('emm', 0.6246851086616516),
 ('prevail', 0.6199260950088501),
 ('calmly', 0.6187276244163513)]

In [62]:
model.wv.most_similar("blackfish")


[('brynden', 0.7911313772201538),
 ('edmure', 0.7663551568984985),
 ('riverrun', 0.7577877044677734),
 ('arnolf', 0.7516084909439087),
 ('surrender', 0.7228758335113525),
 ('diligent', 0.7204763293266296),
 ('fords', 0.7171770930290222),
 ('edmyn', 0.7111344933509827),
 ('craggy', 0.7026949524879456),
 ('tullys', 0.6985017657279968)]

In [63]:
model.wv.doesnt_match(['jon','rikon','robb','arya','sansa','jaime','robert','stanis'])

'robert'

In [64]:
model.wv['tyrion']

array([-0.41962132,  0.44231617,  0.11848823,  0.28249243,  0.6263124 ,
       -0.460915  ,  0.65946895,  0.15889171,  0.00273621, -0.41615862,
        0.31361496,  0.2181349 , -0.52214634,  0.60102314,  0.03325671,
       -0.31180072,  0.4897846 , -0.5835531 , -0.13393272, -0.2043284 ,
        0.28508824,  0.10585408,  0.16903822, -0.16522798, -0.11885287,
       -0.39680225, -0.53733265, -0.15109725, -0.7120651 , -0.32498172,
        0.33472684,  0.19229978, -0.28119156,  0.03423976, -0.51049465,
        0.6028382 ,  0.27737692,  0.04409993, -0.16930573,  0.14558694,
       -0.12396684, -0.09532473, -0.2394649 ,  0.1563585 , -0.18816307,
       -0.20766397, -0.21538974, -0.37147862, -0.11710947,  0.27238527,
       -0.18411654, -0.52252936, -0.3004915 ,  0.4017116 , -0.25110257,
        0.34284458,  0.00073593, -0.24287087,  0.2856193 ,  0.02687889,
        0.72451913,  0.29096934, -0.07442884,  0.32781488, -0.18596947,
        0.0259192 ,  0.4131717 ,  0.22759706,  0.00757495,  0.59

In [65]:
model.wv.similarity('tyrion','jaime')

np.float32(0.6452715)

In [66]:
model.wv.similarity('dragon','jaime')


np.float32(0.30027667)

In [67]:
model.wv.get_normed_vectors().shape

(17310, 100)

In [68]:
y = model.wv.index_to_key

In [69]:
from sklearn.decomposition import PCA

In [70]:
pca  = PCA(n_components = 3)

In [71]:
x = pca.fit_transform(model.wv.get_normed_vectors())

In [72]:
x[:5]

array([[ 0.3192146 ,  0.1898223 , -0.23501164],
       [ 0.38065177,  0.0234279 , -0.04198802],
       [ 0.27473763,  0.2659928 , -0.2904312 ],
       [ 0.06711771,  0.06360567, -0.2885296 ],
       [ 0.32682896,  0.1199919 , -0.0634656 ]], dtype=float32)

In [73]:
import plotly.express as px
fig = px.scatter_3d(
    x=x[:100, 0],
    y=x[:100, 1],
    z=x[:100, 2],
    color=y[:100]
)

fig.show()